# 세션 3 — 하이브리드 검색 (4파서 x 3검색기)

의미 검색(FAISS)은 비슷한 뜻을 잘 찾고, 키워드 검색(BM25)은 정확한 단어를 잘 찾습니다.
둘을 합친 것이 Ensemble입니다. 세션 2의 네 추출본 각각에 세 검색기를 적용해 4 x 3 = 12개 조합을 비교합니다.

```
질문 → FAISS(의미) + BM25(키워드) → Ensemble(합치기)
```

In [1]:
# 준비 셀 — 경로와 API 키 (매 세션 같은 코드로 시작한다)
import os, time, pickle
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()                            # .env 의 OPENAI_API_KEY 를 환경변수로 등록

ROOT = Path.cwd().parent                 # notebooks/ 의 상위 = 프로젝트 폴더
PDF = ROOT / "data" / "저출생_반전을_위한_대책_관계부처_합동_.pdf"
CACHE = ROOT / "cache"
CACHE.mkdir(exist_ok=True)

print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")), "| PDF:", PDF.exists())

OpenAI key: True | PDF: True


In [2]:
# 임베딩 모델과 LLM 준비 (세션 2와 동일)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

# bge-m3 임베딩: 문장 → 1024차원 벡터 (처음 한 번만 약 2GB 다운로드)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},    # 인덱스를 만들 때와 같은 설정이어야 한다
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
# 공통 헬퍼 — 검색 → 프롬프트 → 생성 (세션 2에서 자세히 설명)
RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""

def ask(question, retriever):
    # 질문과 비슷한 청크를 찾아, 그 내용을 근거로 LLM이 답한다
    docs = retriever.invoke(question)
    context = "\n---\n".join(d.page_content for d in docs)
    answer = llm.invoke(RAG_PROMPT.format(context=context, question=question)).content
    return answer, docs

def show_docs(docs, title="검색 결과"):
    # 검색된 청크를 쪽 번호와 함께 보기 좋게 출력한다
    print(title)
    for i, d in enumerate(docs, 1):
        page = d.metadata.get("page", "?")
        print(f"  {i}. (p.{page}) {d.page_content[:100].strip()} ...")

In [4]:
# 정답이 문서에서 확인된 평가 문항 12개 (세션 1에서 만든 것)
EVAL = [
    {"q": "2023년 한국 합계출산율은?",                         "a": "0.72명"},
    {"q": "2023년 출생아 수는?",                               "a": "23만명"},
    {"q": "정부가 2030년까지 목표로 하는 합계출산율은?",        "a": "1.0"},
    {"q": "육아휴직 급여 최대 상한은 얼마로 인상되나?",         "a": "250만원"},
    {"q": "아빠(배우자) 출산휴가 기간은 며칠로 확대되나?",      "a": "20일"},
    {"q": "부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?", "a": "1년 6개월"},
    {"q": "대체인력지원금은 월 얼마로 인상되나?",              "a": "120만원"},
    {"q": "임기 내 공공보육 이용률 목표는?",                   "a": "50%"},
    {"q": "0세반 교사 대 영아 비율은 어떻게 개선되나?",        "a": "1:2"},
    {"q": "'25년 이후 출산가구 신생아 특례대출 소득요건은?",    "a": "2.5억원"},
    {"q": "출산가구 대상 주택공급은 연 몇 호로 확대되나?",      "a": "12만호"},
    {"q": "보고서가 제시한 한국의 육아휴직 소득대체율은?",      "a": "38.6%"},
]

def is_correct(answer, expected):
    # 표기 차이를 흡수한다 ("월 250만원" == "250만원", "20근무일" == "20일") — 세션 1에서 설명
    def norm(s):
        s = s.replace(" ", "").replace(",", "").replace("근무일", "일")
        return s[1:] if s.startswith("월") else s
    return norm(expected) in norm(answer)

## 3-1. 세션 2가 저장한 추출본 불러오기

`cache/splits_all.pkl`(4파서 청크)과 `cache/idx_*`(파서별 FAISS)를 그대로 씁니다. 추출·임베딩을 다시 하지 않습니다.

In [5]:
# 세션 2가 저장해 둔 4개 추출본의 청크를 불러온다 (추출을 다시 하지 않는다)
from langchain_community.vectorstores import FAISS

splits_all = pickle.load(open(CACHE / "splits_all.pkl", "rb"))   # {추출본 이름: 청크 리스트}
print({k: len(v) for k, v in splits_all.items()})

{'pymupdf': 112, '4llm': 122, 'docling': 140, 'vlm': 181}


## 3-2. 파서마다 검색기 세 개 만들기

FAISS는 저장된 인덱스를 로드하고, BM25는 청크로 즉시 만들고, Ensemble은 둘을 5:5로 합칩니다.

In [6]:
# 추출본마다 검색기 3종(FAISS·BM25·Ensemble)을 만든다 → 총 4 x 3 = 12개
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

engines = {}
for parser, sp in splits_all.items():
    # FAISS(의미 검색): 세션 2가 저장한 인덱스를 로드한다
    # 인덱스는 pickle 기반 → 우리가 직접 만든(신뢰할 수 있는) 캐시만 이 옵션으로 연다
    db = FAISS.load_local(str(CACHE / f"idx_{parser}"), embeddings,
                          allow_dangerous_deserialization=True)
    faiss_r = db.as_retriever(search_kwargs={"k": 5})
    # BM25(키워드 검색): 단어가 겹치는 정도로 점수를 매긴다 — 임베딩 없이 청크로 즉시 생성
    bm25_r = BM25Retriever.from_documents(sp); bm25_r.k = 5
    # Ensemble: 두 검색기의 결과를 5:5 가중치로 합쳐 최종 순위를 만든다
    ensemble_r = EnsembleRetriever(retrievers=[faiss_r, bm25_r], weights=[0.5, 0.5])
    engines[parser] = {"FAISS": faiss_r, "BM25": bm25_r, "Ensemble": ensemble_r}
print(list(engines))

['pymupdf', '4llm', 'docling', 'vlm']


## 3-3. 같은 질문, 다른 검색 결과

고유명사가 든 질문으로 FAISS와 BM25가 가져오는 청크를 비교합니다(vlm 기준).

In [7]:
# 같은 질문에 FAISS(의미)와 BM25(키워드)가 서로 다른 청크를 가져오는 것을 눈으로 확인한다
question = "신생아 특례대출 소득요건 완화"
show_docs(engines["vlm"]["FAISS"].invoke(question)[:3], "FAISS")
print()
show_docs(engines["vlm"]["BM25"].invoke(question)[:3], "BM25")

FAISS
  1. (p.2) ```markdown
## 3. 주거 및 결혼·출산·양육

1. **신생아특례대출 소득기준 사실상 한시 폐지**(소득기준 2~25억 원, 3년간)  
   37페이지

2. **출 ...
  2. (p.31) ### ➋ (분양주택 청약 요건 완화) 분양 시 적용되던 결혼, 폐쇄형(청약신청, 무주택 조건, 소득요건) 등 결혼, 메리트로 전환하고, 출산가구 특공 기회도 확대(국토부)

- ...
  3. (p.32) ```markdown
| 구분 | 현행 | 개선 |
|------|------|------|
| 자격 | 생애기간 중 특별공급 1회 당첨 제한 | 신규 출산가구는 특별공급 재당첨 ...

BM25
  1. (p.31) - (공공분양) 일반공급 물량 50% 활용 신설하여 우선공급 신설

  - (공공임대) 일반공급 내 신설하여 우선공급 신설(건설임대), 출산가구 대상 공급 확대(매입, 전세, 재정 ...
  2. (p.31) ```markdown
## ② 양질의 주택 공급 확대

### ➊ (신혼·출산가구 주택공급 확대) 신혼부부 우선공급 확대, GB해제 등을 통한 신규택지 확보, 매입임대 등으로 신혼 ...
  3. (p.29) ```markdown
① (대학등록금 부담 완화) 다자녀 장학금 소득요건을 기존 8구간에서 9구간으로 완화하여 다자녀 가구의 등록금 부담 경감
- 기존 8구간까지 3자녀 이상 지원 ...


## 3-4. 성능 그리드 (12조합 x 12문항)

각 조합으로 12문항을 풀어 점수를 매깁니다. 결과는 `results` 에 저장해 아래 상세에서 다시 씁니다.

> **gpt-4o-mini 144콜 — 몇 분 걸리고 API 과금이 발생합니다.**
> 강의에서는 강사가 셀을 걸어 두고 설명을 진행하니, 수강생은 직접 돌리지 말고
> 저장된 출력을 보면 됩니다. (직접 돌려보고 싶다면 쉬는 시간에!)

In [8]:
# 12개 검색기 조합 x 12문항 = 144콜 채점 (몇 분 — 강사용 셀)
parsers = list(splits_all)
methods = ["FAISS", "BM25", "Ensemble"]
results = {}                              # (추출본, 검색기) → 문항별 (문항, 답변, 정오) 목록
for parser in parsers:
    for m in methods:
        rows = []
        for it in EVAL:
            answer, _ = ask(it["q"], engines[parser][m])
            rows.append((it, answer, is_correct(answer, it["a"])))
        results[(parser, m)] = rows

# 표 출력 — 행: 추출본, 열: 검색기, 칸: 점수
print(f"{'파서':<9}", "".join(f"{m:>11}" for m in methods))
for parser in parsers:
    row = "".join(f"{str(sum(ok for _,_,ok in results[(parser,m)]))+'/12':>11}" for m in methods)
    print(f"{parser:<9}", row)

파서              FAISS       BM25   Ensemble
pymupdf         12/12       9/12      12/12
4llm            11/12       9/12      11/12
docling         11/12       9/12      11/12
vlm              9/12       6/12       8/12


## 3-5. 한 조합 골라 문항별로 보기

`PARSER`, `METHOD` 를 바꿔 가며 어떤 질문을 맞고 틀렸는지, 실제 답변은 무엇이었는지 확인합니다.

In [9]:
# 조합 하나를 골라 문항별 정오와 실제 답변을 본다 (PARSER, METHOD 를 바꿔 가며 탐색)
PARSER, METHOD = "vlm", "Ensemble"

print(f"[{PARSER} x {METHOD}]")
for it, ans, ok in results[(PARSER, METHOD)]:      # 저장해 둔 results 재사용 — 추가 호출 없음
    print(f"{'O' if ok else 'X'} {it['a']:<8} | {ans[:70]}")

[vlm x Ensemble]
O 0.72명    | 2023년 한국의 합계출산율은 0.72명입니다.
O 23만명     | 2023년 출생아 수는 23만명입니다.
O 1.0      | 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
O 250만원    | 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.
O 20일      | 아빠(배우자) 출산휴가 기간은 10일에서 20일로 확대됩니다.
O 1년 6개월   | 부모가 모두 3개월 이상 육아휴직을 사용할 경우, 총 육아휴직 기간은 1년 6개월로 연장됩니다.
X 120만원    | 대체인력지원금은 월 20만원으로 인상됩니다.
X 50%      | 임기 내 공공보육 이용률 목표는 명시되어 있지 않습니다. 그러나 정책 목표는 초저출생 추세 반전을 위한 다양한 돌봄 서비스 확
O 1:2      | 0세반 교사 대 영아 비율은 현행 1:3에서 1:2로 개선될 예정입니다.
X 2.5억원    | '25년 이후 출산가구 신생아 특례대출 소득요건은 사실상 한시 폐지되어 소득기준이 2~25억 원으로 설정됩니다.
O 12만호     | 출산가구 대상 주택공급은 연간 12만호 + α로 확대됩니다.
X 38.6%    | 보고서에 따르면 한국의 육아휴직 소득대체율은 통상임금의 80%로, 현재 월 150만원의 상한이 설정되어 있습니다.


## 3-6. 그리드 분석

| 파서 | FAISS | BM25 | Ensemble |
|---|---|---|---|
| pymupdf | 12 | 9 | 12 |
| 4llm | 11 | 9 | 11 |
| docling | 11 | 9 | 11 |
| vlm | 9 | 6 | 8 |

행(추출)이 열(검색)보다 영향이 큽니다. pymupdf 행은 전부 높고 vlm 행은 전부 낮습니다.
어떤 검색기를 써도 추출이 나쁘면 한계가 생깁니다.

BM25 열이 파서와 무관하게 낮은 것(9/9/9/6)은 한국어 때문입니다. BM25 기본은 띄어쓰기로 단어를 나누는데,
한국어는 "합계출산율은", "출생아수는"처럼 조사가 붙어 매칭이 약합니다. Kiwi 같은 형태소 분석기를 넣으면 크게 좋아집니다 (아래 심화 1에서 확인).

Ensemble은 대체로 FAISS와 비슷합니다. 다만 vlm처럼 한쪽(BM25 6)이 많이 약하면 노이즈 청크가 섞여
실행에 따라 FAISS보다 낮게 나오기도 합니다. 한쪽이 크게 약할 때는 가중치(weights) 조정이 처방입니다.

`2.5억` 을 `2~25억` 으로 답하는 실패는 세션 2의 VLM 추출 오독이 청크에 박힌 것이라, 어떤 검색기로도 고칠 수 없습니다.
검색은 추출이 남긴 한계를 넘지 못합니다 (세션 4에서 이어서 다룹니다).

## 직접 해보기

> 예시 코드를 **새 셀에 복사해** 실행해 보세요.

**1. 3-5의 `PARSER`/`METHOD` 를 바꿔 가장 낮은 조합이 어디서 틀리는지 보세요.**
```python
PARSER, METHOD = "vlm", "BM25"        # 가장 낮았던 6/12 조합
print(f"[{PARSER} x {METHOD}]")
for it, ans, ok in results[(PARSER, METHOD)]:
    print(f"{'O' if ok else 'X'} {it['a']:<8} | {ans[:60]}")
```

**2. Ensemble `weights` 를 [0.7, 0.3] 으로 바꾸면 답이 달라질까요?**
```python
ens_73 = EnsembleRetriever(
    retrievers=[engines["vlm"]["FAISS"], engines["vlm"]["BM25"]],
    weights=[0.7, 0.3])               # FAISS 쪽에 힘을 실어 준다
print(ask("대체인력지원금은 월 얼마로 인상되나?", ens_73)[0][:80])
```

**3. BM25에 한국어 형태소 분석기(Kiwi)를 넣으면 점수가 오르는지** — 아래 심화에 완성 코드가 있습니다.
먼저 스스로 시도해 보세요. (힌트: `BM25Retriever.from_documents(sp, preprocess_func=...)`)

In [13]:
PARSER, METHOD = "vlm", "BM25"        # 가장 낮았던 6/12 조합
print(f"[{PARSER} x {METHOD}]")
for it, ans, ok in results[(PARSER, METHOD)]:
    print(f"{'O' if ok else 'X'} {it['a']:<8} | {ans[:60]}")

[vlm x BM25]
X 0.72명    | 2023년 한국의 합계출산율에 대한 정보는 제공되지 않았습니다. 맥락에서 언급된 내용은 미국, 일본, 이스라
X 23만명     | 2023년 출생아 수는 19.4만건입니다.
O 1.0      | 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
O 250만원    | 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.
O 20일      | 아빠(배우자) 출산휴가 기간은 10일에서 20일로 확대됩니다.
O 1년 6개월   | 부모가 모두 3개월 이상 육아휴직을 사용할 경우, 총 육아휴직 기간은 1년 6개월로 연장됩니다.
X 120만원    | 제공된 맥락에는 대체인력지원금의 인상 금액에 대한 정보가 포함되어 있지 않습니다. 따라서 대체인력지원금이 월
X 50%      | 임기 내 공공보육 이용률 목표에 대한 구체적인 내용은 맥락에 포함되어 있지 않습니다. 따라서 해당 정보를 제
O 1:2      | 0세반 교사 대 영아 비율은 1:3에서 1:2로 개선될 예정입니다.
X 2.5억원    | 제공된 맥락에는 '25년 이후 출산가구 신생아 특례대출 소득요건'에 대한 구체적인 정보가 포함되어 있지 않습
O 12만호     | 출산가구 대상 주택공급은 연간 12만호 + α로 확대됩니다.
X 38.6%    | 보고서에 따르면 한국의 육아휴직 소득대체율은 통상임금의 80%로, 월 150만원이 상한입니다.


## 시간이 남으면 (심화 1) — Kiwi 형태소 분석기로 BM25 살리기

BM25가 낮았던 이유는 토크나이저입니다. 기본은 띄어쓰기 단위라 **"합계출산율은"과 "합계출산율"을
다른 단어로 봅니다.** 형태소 분석기 Kiwi 는 조사를 떼어내 "합계+출산+율+은"으로 나누므로 매칭이 살아납니다.

먼저 두 토크나이저가 같은 문장을 어떻게 다르게 나누는지 눈으로 확인합니다. (API 호출 없음)

In [10]:
# Kiwi 형태소 분석기 — 조사가 붙은 한국어 단어를 의미 단위로 쪼갠다
from kiwipiepy import Kiwi

kiwi = Kiwi()

def kiwi_tokenize(text):
    # 문장을 형태소 리스트로 바꾼다 (BM25의 토크나이저로 끼울 함수)
    return [t.form for t in kiwi.tokenize(text)]

sentence = "2023년 한국 합계출산율은 0.72명이다"
print("기본(띄어쓰기):", sentence.split())
print("Kiwi(형태소)  :", kiwi_tokenize(sentence))
# 질문 "합계출산율은?"의 '합계출산율'이 기본 토크나이저로는 문서의 '합계출산율은'과 매칭되지 않지만,
# Kiwi로는 둘 다 [합계, 출산, 율]로 쪼개져 매칭된다.

기본(띄어쓰기): ['2023년', '한국', '합계출산율은', '0.72명이다']
Kiwi(형태소)  : ['2023', '년', '한국', '합계출산율', '은', '0.72', '명', '이', '다']


In [11]:
# Kiwi 토크나이저를 단 BM25로 pymupdf 추출본을 다시 채점한다 (12콜, 1분 안팎)
# preprocess_func: BM25가 문서·질문을 단어로 쪼갤 때 쓸 함수를 우리 것으로 교체한다
bm25_kiwi = BM25Retriever.from_documents(splits_all["pymupdf"], preprocess_func=kiwi_tokenize)
bm25_kiwi.k = 5

correct = 0
for it in EVAL:
    answer, _ = ask(it["q"], bm25_kiwi)
    ok = is_correct(answer, it["a"]); correct += ok
    print(("O" if ok else "X"), it["a"], "->", answer[:45].replace("\n", " "))

# 기본 토크나이저 점수(3-4 그리드에 저장된 것)와 나란히 비교
before = sum(ok for _, _, ok in results[("pymupdf", "BM25")])
print(f"\nBM25(기본) {before}/12  →  BM25(Kiwi) {correct}/12")

O 0.72명 -> 2023년 한국의 합계출산율은 0.72명입니다.
O 23만명 -> 2023년 출생아 수는 23만명입니다.
O 1.0 -> 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
O 250만원 -> 육아휴직 급여 최대 상한은 250만원으로 인상됩니다.
O 20일 -> 아빠(배우자) 출산휴가 기간은 10일에서 20일로 확대됩니다.
O 1년 6개월 -> 부모가 모두 3개월 이상 육아휴직을 사용하면 총 육아휴직 기간이 1년에서 1년 6
O 120만원 -> 대체인력지원금은 월 120만원으로 인상됩니다.
O 50% -> 임기 내 공공보육 이용률 목표는 50%입니다. 현재 이용률은 40%입니다.
O 1:2 -> 0세반 교사 대 영아 비율은 1:3에서 1:2로 개선될 예정이다.
O 2.5억원 -> '25년 이후 출산가구의 신생아 특례대출 소득요건은 2.5억원으로 완화됩니다.
O 12만호 -> 출산가구 대상 주택공급은 연 12만 호로 확대됩니다.
O 38.6% -> 한국의 육아휴직 소득대체율은 38.6%입니다.

BM25(기본) 9/12  →  BM25(Kiwi) 12/12


## 시간이 남으면 (심화 2) — Multi-Query: 질문을 여러 개로 바꿔 검색

사용자가 물은 한 문장이 문서의 표현과 다르면 검색이 빗나갑니다.
**Multi-Query**는 LLM에게 질문을 3~4가지로 바꿔 쓰게 한 뒤, 각 변형으로 검색해 결과를 합칩니다.
검색당 LLM 1콜이 추가되는 대신 표현 차이에 강해집니다.

In [12]:
# Multi-Query — LLM이 질문을 여러 변형으로 바꿔 쓰고, 각 변형으로 검색해 결과를 합친다
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever

# LLM이 만든 질문 변형을 눈으로 보기 위해 로깅을 켠다
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

mq = MultiQueryRetriever.from_llm(retriever=engines["pymupdf"]["FAISS"], llm=llm)

question = "아이 맡길 곳이 없으면 어떤 지원을 받을 수 있나?"   # 문서 표현과 일부러 다르게 물어본다
docs = mq.invoke(question)                # 내부에서 변형 생성(LLM 1콜) + 변형별 검색이 실행된다
show_docs(docs[:7], f"Multi-Query 검색 결과 ({len(docs)}개 중 7개)")

INFO:langchain.retrievers.multi_query:Generated queries: ['아이를 맡길 곳이 없을 때 어떤 종류의 지원을 받을 수 있을까요?', '아이를 돌볼 수 있는 장소가 없을 경우, 어떤 도움을 받을 수 있는지 알고 싶습니다.', '아이를 맡길 수 있는 옵션이 없을 때, 어떤 지원 서비스가 제공되나요?']


Multi-Query 검색 결과 (7개 중 7개)
  1. (p.20) - 34 -
   - 대체인력 확보가 어려운 지역, 업종에 외국인 근로자(고용허가업종에 한정), 
외국인 유학생(졸업생) 등을 대체인력으로 공급(고용부, 법무부)
 ➋ (고용보험 ...
  2. (p.24) - 42 -
    - 지자체 대상 공모를 통해 중소기업 근로자가 출퇴근시 아이를 
맡기기 편한 장소에 상생형 공동어린이집 설치·운영(고용부)
    - 직장어린이집을 위탁운영하는 ...
  3. (p.33) 과 양육을 포기하지 않도록 ‘위기임산부 상담체계’ 신설(복지부)
  위기임산부가 직접 양육을 우선 고려할 수 있도록 출산·양육지원 제도 
상세 안내, 필요시 국내입양 등 아동보호 ...
  4. (p.22) - 38 -
2
 국가책임 교육·돌봄체계 완결을 통한 양육부담 획기적 해소
◇ 양육은 공동체 책임이라는 원칙하에 Parental Care에서 Public Care로 
전환하고, 누 ...
  5. (p.26) (부천 - 출퇴근 시간 아이돌봄) 출퇴근시간(8~10시, 17~20시)에 육아나눔터에서 
5세～초등 3학년 아동 대상 시니어클럽 어르신이 참여한 돌봄 제공
(증평군 - ‘행복돌 ...
  6. (p.32) 되면 정부지원도 없어진다고 합니다. 아이를 갖고 싶은 부부에게 정부가 확실히 
지원해 주는 것이 저출생 대책이라고 생각합니다.(정책수요자 자문단 위촉식)
▸ “아이가 안 생겨서 입 ...
  7. (p.2) √ 야간연장(5:30~24시 이용 가능) 및 휴일어린이집 확대
√ 방학에도 늘봄학교 운영
√ 지자체 돌봄 연계를 통해 방학 중 돌봄공백 대응
    ↳ 방학 중: 지역아동센터(12 ...


## 정리
- 추출 x 검색 조합에 따라 성능이 달라집니다. 그리드로 한눈에 봅니다.
- Ensemble이 보통 안정적이지만 항상 최고는 아닙니다(가중치·토크나이저 영향).
- 한국어 BM25는 형태소 분석기(Kiwi)가 처방입니다 — 심화에서 9/12 → 12/12로 올랐습니다.
- 일부 실패는 추출 단계 문제라 검색으로는 고칠 수 없습니다 → 다음 세션.